In [1]:
import requests
import json

lat, lon = 44.204957, -72.518160
radius_m = 3000

query = f"""
[out:json][timeout:60];
(
  way["building"](around:{radius_m},{lat},{lon});
  relation["building"](around:{radius_m},{lat},{lon});
);
out body;
>;
out skel qt;
"""

response = requests.post("https://overpass-api.de/api/interpreter", data=query)
data = response.json()

# Convert to GeoJSON
features = []
nodes = {el['id']: el for el in data['elements'] if el['type'] == 'node'}

for el in data['elements']:
    if el['type'] == 'way' and 'tags' in el and 'building' in el['tags']:
        coords = [[nodes[n]['lon'], nodes[n]['lat']] for n in el['nodes'] if n in nodes]
        if coords:
            features.append({
                "type": "Feature",
                "properties": el.get('tags', {}),
                "geometry": {"type": "Polygon", "coordinates": [coords]}
            })

geojson = {"type": "FeatureCollection", "features": features}

with open("buildings_3km.geojson", "w") as f:
    json.dump(geojson, f)

print(f"Downloaded {len(features)} buildings")

Downloaded 4641 buildings
